# Federated Learning Intrusion Detection System - Complete Thesis Experiments
## End-to-End Execution Notebook (No Mock Data)

In [ ]:
# INSTALL DEPENDENCIES
!pip install torch pandas scikit-learn imbalanced-learn matplotlib seaborn torch_geometric -q

In [ ]:
# IMPORTS
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")

## 1. Data Loading and Preprocessing

In [ ]:
# Define columns for NSL-KDD
COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land',
    'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'label', 'difficulty_level'
]

def load_and_preprocess_nslkdd(train_path, test_path):
    print(f"Loading data from {train_path} and {test_path}...")
    train_df = pd.read_csv(train_path, header=None, names=COLUMNS)
    test_df = pd.read_csv(test_path, header=None, names=COLUMNS)
    
    # Drop difficulty_level
    if 'difficulty_level' in train_df.columns:
        train_df.drop('difficulty_level', axis=1, inplace=True)
    if 'difficulty_level' in test_df.columns:
        test_df.drop('difficulty_level', axis=1, inplace=True)
    
    # Binary label mapping
    train_df['label'] = train_df['label'].astype(str).apply(lambda x: 0 if x == 'normal' else 1)
    test_df['label'] = test_df['label'].astype(str).apply(lambda x: 0 if x == 'normal' else 1)
    
    y_train = train_df['label'].values
    y_test = test_df['label'].values
    
    X_train = train_df.drop('label', axis=1)
    X_test = test_df.drop('label', axis=1)
    
    categorical_cols = ['protocol_type', 'service', 'flag']
    numerical_cols = [c for c in X_train.columns if c not in categorical_cols]
    
    # Force numerical types
    for col in numerical_cols:
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
    
    X_train.fillna(0, inplace=True)
    X_test.fillna(0, inplace=True)
    
    # One-Hot Encoding
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_train_cat = encoder.fit_transform(X_train[categorical_cols])
    X_test_cat = encoder.transform(X_test[categorical_cols])
    
    # Standard Scaling
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train[numerical_cols])
    X_test_num = scaler.transform(X_test[numerical_cols])
    
    X_train_final = np.hstack([X_train_num, X_train_cat])
    X_test_final = np.hstack([X_test_num, X_test_cat])
    
    print(f"Preprocessing complete. Train shape: {X_train_final.shape}, Test shape: {X_test_final.shape}")
    return X_train_final, X_test_final, y_train, y_test

# Load data
X_train, X_test, y_train, y_test = load_and_preprocess_nslkdd('data/KDDTrain+.csv', 'data/KDDTest+.csv')

In [ ]:
# Create DataLoader helper
def create_dataloader(X, y, batch_size=64, shuffle=True):
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.LongTensor(y)
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

# Partition data for FL
def partition_data_for_fl(X, y, num_clients=5, iid=True):
    if iid:
        indices = np.random.permutation(len(X))
        X_shuffled, y_shuffled = X[indices], y[indices]
        split_size = len(X) // num_clients
        partitions = []
        for i in range(num_clients):
            start = i * split_size
            end = (i + 1) * split_size if i < num_clients - 1 else len(X)
            partitions.append((X_shuffled[start:end], y_shuffled[start:end]))
        return partitions
    else:
        # Non-IID: shard by label
        partitions = [[] for _ in range(num_clients)]
        unique_labels = np.unique(y)
        for label in unique_labels:
            idx = np.where(y == label)[0]
            np.random.shuffle(idx)
            shards = np.array_split(idx, num_clients)
            for i, shard in enumerate(shards):
                partitions[i].append(shard)
        
        final_partitions = []
        for i in range(num_clients):
            all_idx = np.concatenate(partitions[i])
            final_partitions.append((X[all_idx], y[all_idx]))
        return final_partitions

# Create partitions
np.random.seed(42)
partitions = partition_data_for_fl(X_train, y_train, num_clients=5)
print(f"Created {len(partitions)} client partitions")

## 2. Model Definition

In [ ]:
class IDS_NeuralNet(nn.Module):
    def __init__(self, input_dim, hidden_dim_1=64, hidden_dim_2=32, dropout_rate=0.3):
        super(IDS_NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim_1)
        self.bn1 = nn.BatchNorm1d(hidden_dim_1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(hidden_dim_1, hidden_dim_2)
        self.bn2 = nn.BatchNorm1d(hidden_dim_2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.out = nn.Linear(hidden_dim_2, 1)  # Binary classification with BCE
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        x = self.out(x)
        return x
    
    def get_weights(self):
        return self.state_dict()
    
    def set_weights(self, state_dict):
        self.load_state_dict(state_dict)

input_dim = X_train.shape[1]
global_model = IDS_NeuralNet(input_dim)
print(f"Model initialized with input dim: {input_dim}")

## 3. Federated Client and Server

In [ ]:
class FederatedClient:
    def __init__(self, client_id, dataloader, device, model):
        self.client_id = client_id
        self.dataloader = dataloader
        self.device = device
        self.model = IDS_NeuralNet(input_dim)
        self.model.set_weights(model.get_weights())
        self.model.to(device)
        
    def train_local(self, epochs=5, lr=0.001, apply_smote=False, mu=0.0):
        self.model.train()
        
        # Extract data for SMOTE
        all_X, all_y = [], []
        for X_batch, y_batch in self.dataloader:
            all_X.append(X_batch.numpy())
            all_y.append(y_batch.numpy())
        
        X_all = np.vstack(all_X)
        y_all = np.concatenate(all_y)
        
        # Apply SMOTE if requested and both classes exist
        if apply_smote and len(np.unique(y_all)) > 1:
            try:
                k_neighbors = min(5, sum(y_all == 1) - 1, sum(y_all == 0) - 1)
                if k_neighbors > 0:
                    smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
                    X_all, y_all = smote.fit_resample(X_all, y_all)
                    print(f"Client {self.client_id}: SMOTE applied. New size: {len(y_all)}")
            except Exception as e:
                print(f"Client {self.client_id}: SMOTE failed ({e}), using original data")
        
        train_loader = create_dataloader(X_all, y_all, batch_size=64, shuffle=True)
        
        # Class weights for imbalanced data
        pos_weight = torch.tensor([sum(y_all == 0) / max(sum(y_all == 1), 1)]).to(self.device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
        
        global_weights = {k: v.clone() for k, v in self.model.state_dict().items()}
        
        for epoch in range(epochs):
            total_loss = 0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(X_batch).squeeze()
                loss = criterion(outputs, y_batch.float())
                
                # FedProx regularization
                if mu > 0:
                    prox_term = 0
                    for name, param in self.model.named_parameters():
                        prox_term += torch.norm(param - global_weights[name])**2
                    loss += (mu / 2) * prox_term
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            scheduler.step(total_loss / len(train_loader))
        
        return self.model.get_weights(), len(y_all)

class FederatedServer:
    def __init__(self, global_model, device):
        self.global_model = global_model
        self.device = device
        self.global_model.to(device)
        
    def aggregate(self, client_weights, client_sample_counts, optimizer_type='fedavg'):
        total_samples = sum(client_sample_counts)
        aggregated_state = {}
        
        for key in client_weights[0].keys():
            weighted_sum = torch.zeros_like(client_weights[0][key], dtype=torch.float64)
            for i, client_state in enumerate(client_weights):
                weight = client_sample_counts[i] / total_samples
                weighted_sum += weight * client_state[key].to(torch.float64)
            aggregated_state[key] = weighted_sum.to(client_weights[0][key].dtype)
        
        return aggregated_state
    
    def evaluate(self, test_loader):
        self.global_model.eval()
        all_preds, all_labels, all_losses = [], [], []
        criterion = nn.BCEWithLogitsLoss()
        
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                outputs = self.global_model(X_batch).squeeze()
                loss = criterion(outputs, y_batch.float())
                all_losses.append(loss.item())
                
                predicted = (torch.sigmoid(outputs) > 0.5).long()
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())
        
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        
        return {
            'accuracy': accuracy_score(all_labels, all_preds),
            'precision': precision_score(all_labels, all_preds, zero_division=0),
            'recall': recall_score(all_labels, all_preds, zero_division=0),
            'f1_score': f1_score(all_labels, all_preds, zero_division=0),
            'loss': np.mean(all_losses),
            'confusion_matrix': confusion_matrix(all_labels, all_preds)
        }

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 4. Run Experiments

In [ ]:
def run_experiment(name, use_smote=False, mu=0.0, num_rounds=20):
    print(f"\n{'='*60}")
    print(f"Running Experiment: {name}")
    print(f"{'='*60}")
    
    global_model = IDS_NeuralNet(input_dim)
    server = FederatedServer(global_model, device)
    
    history = {'accuracy': [], 'f1_score': [], 'recall': [], 'loss': []}
    test_loader = create_dataloader(X_test, y_test, batch_size=64, shuffle=False)
    
    for round_idx in range(1, num_rounds + 1):
        client_weights = []
        client_samples = []
        
        for i, (X_c, y_c) in enumerate(partitions):
            client_loader = create_dataloader(X_c, y_c, batch_size=64, shuffle=True)
            client = FederatedClient(i, client_loader, device, global_model)
            weights, samples = client.train_local(epochs=5, lr=0.001, apply_smote=use_smote, mu=mu)
            client_weights.append(weights)
            client_samples.append(samples)
        
        aggregated = server.aggregate(client_weights, client_samples)
        global_model.load_state_dict(aggregated)
        
        metrics = server.evaluate(test_loader)
        history['accuracy'].append(metrics['accuracy'])
        history['f1_score'].append(metrics['f1_score'])
        history['recall'].append(metrics['recall'])
        history['loss'].append(metrics['loss'])
        
        print(f"Round {round_idx}/{num_rounds} - Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1_score']:.4f}, Loss: {metrics['loss']:.4f}")
    
    return history

# Run experiments
results = {}
results['Baseline'] = run_experiment('Baseline (FedAvg)', use_smote=False, mu=0.0, num_rounds=20)
results['SMOTE+Weights'] = run_experiment('SMOTE + Class Weights', use_smote=True, mu=0.0, num_rounds=20)
results['FedProx'] = run_experiment('FedProx + SMOTE', use_smote=True, mu=0.01, num_rounds=20)

print("\n✓ All experiments completed!")

## 5. Visualization

In [ ]:
# Plot results
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
for name, hist in results.items():
    plt.plot(hist['accuracy'], label=name)
plt.title('Accuracy over Rounds')
plt.xlabel('Round')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
for name, hist in results.items():
    plt.plot(hist['f1_score'], label=name)
plt.title('F1-Score over Rounds')
plt.xlabel('Round')
plt.ylabel('F1-Score')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 3)
for name, hist in results.items():
    plt.plot(hist['recall'], label=name)
plt.title('Recall over Rounds')
plt.xlabel('Round')
plt.ylabel('Recall')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 4)
for name, hist in results.items():
    plt.plot(hist['loss'], label=name)
plt.title('Loss over Rounds')
plt.xlabel('Round')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/experiment_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Plots saved to plots/experiment_results.png")

In [ ]:
# Final comparison table
final_metrics = []
for name, hist in results.items():
    final_metrics.append({
        'Method': name,
        'Accuracy': hist['accuracy'][-1],
        'Precision': 'N/A',  # Would need to store separately
        'Recall': hist['recall'][-1],
        'F1-Score': hist['f1_score'][-1],
        'Final Loss': hist['loss'][-1]
    })

df_results = pd.DataFrame(final_metrics)
print("\nFinal Results Comparison:")
print(df_results.to_string(index=False))